# FEVER — original versus trained RoBERTa on all 19,998 claims
**Select T4 GPU, then Runtime → Run all. Evaluation is enabled; no training runs.**

The last-3-block model was chosen using the separate training-pool validation
set: macro-F1 **69.42%**, versus 68.24% (last 2) and 66.30% (last 1). Those are
validation F1 scores, not benchmark accuracy. This notebook rechecks selection
and verifies that the final weights match the selected checkpoint at step 11,000.

For every official FEVER development claim, prepare one temporary full-article
RAG database and pass the same retrieved passages to the original and selected
RoBERTa models. Both use the same 512-token limit, and retained input text is
checked for equality. Top 10 existing ranked Wikipedia URLs → first 5 articles
→ 200-token chunks → MiniLM embeddings → top 30 vector candidates → 7,600-character
context. Full articles and vectors are deleted after each claim. Only source
metadata, supplied passages and model traces are retained.

MiniLM embeddings run in FP32 on the GPU to reduce the previous CPU bottleneck.
This is a separately recorded evaluation protocol; floating-point differences
from previous CPU embeddings are possible. Both classifier conditions share the
new evidence exactly. No speedup is claimed before measuring it.

The target is all **19,998**, not the 90-claim diagnostic. The run can require
multiple sessions because of live article fetching. A measured ETA appears
after several completed claims. Use Run all again to resume the same folder.
The previously inspected official development set is not an untouched hidden test.

## 1. Check GPU

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

## 2. Install runtime libraries

In [2]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.


## 3. Connect Drive and verify the original bundle
Your training notebook, model checkpoints and earlier predictions are preserved.

In [4]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/FEVER_Project")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
locations = [DRIVE_PROJECT / "FEVER_Colab_Bundle.zip",
             Path("/content/drive/MyDrive/FEVER_Colab_Bundle.zip")]
archive = next((p for p in locations if p.is_file()), None)
if archive is None:
    raise FileNotFoundError("The FEVER_Project folder is ready in My Drive. Upload FEVER_Colab_Bundle.zip into it, wait for the upload to finish, then rerun this cell. No path needs typing.")
print("Found your data:", archive)
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)
# Install the debug runtime embedded in this notebook; the original Drive ZIP is unchanged.
import base64, zlib, hashlib, importlib
patch_source = zlib.decompress(base64.b64decode('eJy9fety20bW4H89BZypnQZsCKLsxBuTQaYcWc74G99WtjO7xbDwgSQoISIBDgBKVjyq2qfZB9sn2XPrC0CQUjyzX2rGIoC+9+lzP6e/+eabk3KZTr10nq6brPIWZeVln/O6yYtzb7FZLg/Tqslny8w7e/5z5L0tveuyuvSg1Ly8LpZlOq+9svDy1bqsmujg4F2ReT+//+Stynm29NLGS70mX2UjKLHaNOkUGlpn1eFsmeYrb3aRzS7XZV401MiLKr/KIu/jRV578L/UK7Lrg7++PPr2cJo3MCqoCE0VTegVJTZcZetleuOVC6+5yLwsrZY5zODdcpmuUu/7n7xVltabKsMq0cE333xzsKhK6LRcLrNZk5dFLcP2TspNAZPn7/O0yXDI+qN+DvGf38siO5AP57PwIq0vlvk05DfwK1plDSxlk4a/1WURrtLmIizrsEqLebkKK2gkW60X+ZJb4w7XUAiq6v7ew+PB6+c/nb7+EI/Vh0/v3787+/hBhers9OWnj6f46+27j97p23effv6r9+rty3dqcvDm3Qss/0X94zor1FD9D/hzhP88jr47fPLT4auibqrNrIHKtDxQBEd6SA9Hr/HfwyfR41bR24ODebbw6os0wRH7OM5geOBdxDLtCD49/u6pHxx413lzQSPnUlG5zgpfVVMVeGntLaAWQRZMEmAFltqHHqfzdLiIqiyd+8eDx98+xH+CcKpUMLyINmtcd38KbVdZs6kK7yK6yD7P8/OsbqBHGll+XqTwLfPL6W/BUJdrDw73IZpvVusaS4U1LHFymd3U8cdqk4VZgRCSpPUsz+OX6bLOgigrZgC8fhBsdYhjTaryupa1kB7H1AedBb8OaKI1TtRZD6rZZJ+hpaheL3OAlSKr/cDLYRoRLHi+9oMJdVKU1Spd5r9n0JbuQXkq+g3OiV9Hs7TOFuVyrhuCcUq1a9+U1zAb4VsNt9GmmQVRXpcL7EHP6bqC/UjKYsYbHF6ly02G2wwH5v1muszrCzgyq/UyazLvPz68ewsndXkDmCC7gsNGR3CWwWHUZebeusrmOR2wo1lZ4KQjPHwE57FdkxH+G63TCo/n6nKeVz4/6K1BLJSUl/QEYAArRTXoPSwdwhS8c9aePjsrHTyIaTbDKs3rzPsFf59WVVn56lSjOIuWEM8t0hkc+HyxyKp66KlHsDE8WOhf4PDAwyMcU18I9UmRrjJfReoRvcKnRyoC8J9D80ofDawjh+LaORMGOH0aaLjoA8jRIlosN/WFH4zKOlrUN8XMh1dwJIsSd9+Dt7IPPvYT0ohpcxdVlgEgIbxkVbiuyqYE9IdLx694O/g39EN/794L2rmYSx8p3WqEk1FfuU+6kZ6tei+f9L5E3llWbwA/88IC5i+r/Dwv0qVXZw1uah0ifZpdlGWNkPn29O9etSk83JoIdySDVR124d4sjkE4Fr3Yb7ioAPf54iaZboo5YMWqLBucI/7l5aQ3o1XszJjeHalVWuQLQCe8VC2sEBwQ3sAxhoxzEIOsxgr3uVaTCEa7sqtpkDI1jLVgEblezxL+RENlcgtL5y1SqDpHAKeaOPjrOrbYjUeLZOwoA8BM8mK9aWoa9VIFoypb1D2zs+WBoYA17p8lDB4A18d+YMjHz549+x53C999qcYqn6sJrUSF88dSt26xL3gipVjQKfcghv33cXBB74lfA9UH9JQul97xs5AaXCzyWQ6AMwdktizXyCl4r17UiscpbAE1GdEBrQlY9fsvy+FT+I+GscRhMNW+7ev+U5HpAZheaZkAcUJrusu0uPEVvVc0L5yzOoeT5jxmV/kcaFQmr1qL0DtzHHqK+NhbZATQtYeIOYU6vFNHaQEcFRWhgdAm4o7FXyw1glUnrm174QElQq9FPwDRp2SdL8vGABA0318YPuRzGke7xgEe7Toe3zWcz7f0+zP+HlPPSM8mE1pbbGI8mPyZ/h5PcC1xues/m+la0o1l+tbyre5/Liws0WCvBJywTNc40HUF7KyvfkEkkcNuM5oYapDL7GZQA/VI9gC6BJQLdHvk0dR5TQGbOYsCKLBGNqAhNCZoCtcwRBBldg2PZHKxSJryEo4U4oumuiH+C3nN87I8X2bRjHh+YTg3dVbh6hsSZ95E53Cg1F9fJh/f/e30LUHG51m2brxT+gMj0gwHkKCsuMorwAndOsw4rUrAtpo5hhO1LmHM/4ATBSciqQCka2gtfgtsClEnHOvF5hzQ+jnQ5Cy52Jjh/nXxfJ3zrPJiUcb0DNwQSRwJvtLNS6Pb/YSEVM1wmMbRgsWd5euZMhJvzVKcwdEF7kqA44SOkZfOZlldA3LFYQAv4L0EqCIWG+jSlI6dSEdc8KbcVHW2xMMP2Gk+9/TiISCydPYhm8FC1yR/pZumBPYtn3nLfAbsQkbtrJsUUELk0VC85madIXrHvz6wtAlxKUkS8MriMsORAjCOgRz4PmxZ2gBirUOFL1U4CPB0DBxe1seFjeocGEIgsPh1PBHutSJWBIkrMD01UmVgh+p0AcxgAew2YBRzLGhjlN4GFXKjF2morrP8/KIxQqWXroHofs5hotnyRoUVIMm5j4M7evzw4ZNB+DgI1c/5T94UmOEq81hErHg7vH9sUvjxO50Z56R8Udi/GnZGMbSjsKNO9EiS6U0DBHiIfd8SMAugI7Gg8yVw2ZTV7AK53NllCnQ4/rIebouFESAK7NRf89quCVMpqguyGZz5okb2HErBI27sMkOkAA8wwRoghIcTqgK4RlgYBYziZd4cLkH8hQVVQNrr+nC23qjJrTNxPSo11L9CdQ6FhtRzNNvM6bAnQAgBqJinHdD+OgXyOkmvgHVAdhlEF2SjCJRChZ91WzJDqsILRnxHgs8965UvSJrf2U3vSfuQoQivX3o/eicXaXGeGQBAuIe3H79FXUSITCLgyQyZQEDsmzWdI1RueCfvPxFvmAP+bUACyOaRg8Sh8lCFe5coVL+cPX/jASgONZT2lwdwXmcgYgAPMQiiBsjtEhDQqqxuBKKPA40rYS/rrGepzmeRKC985hZ2L5rzBSSC5iaZpbALUO3g7NPbj6/enCZnp7+8+vAKxLnYU0QODufZdHN+ePWdOjgA0gSICZb3TVnk0BaLgyfAHpSowKnK8wox1xrEEm++qUh+wunRiUPOpjwHwlaQiAi4Dilh7ciFdXRw8BEY9ynA4jktGmwRMgS88LQXsJsZkNQaSChJst6rBneJNT+MFS+zbH0IxPEqY/WOh8uXAAnImyTxEaWGIvkAuzPPseeQFj40wmo8ABxEBBZ+XcAZaqbAIiV1hjXq+PEgrC/K60TPmCnFBjqIhftghly2ieYA+BFeYe9Rn5DlfLiP1MsNmfHH5hc3JEAb0xLBBiySmTCngSlwniW6mFtHt70961X62V8A7mv8rW9BGA2+0023Vqb1pNtelrPL2CxLdPYanu3IynUCEFI0TonTK8KqXAIhi8AlHk/4zTStYiZe3H4NfSGzCNuHggGtAv6TFMipj2QM5wkJrM7KH/lKVwUhv90SUGzDdUovDQxvFbdaE2H+swpJXQRjj9WmWRx+71YDxB1/UbZpoCLtvgBz6v1UQwukiqAUESoCqzLQimXkZ6gEbhMpK48h9O7+p3AUG8D7CtAcspNIMhAm4A0BAr8iYKYRMkLXQyyy64T5VDUcQDFYCGSkNs1MDUnjFCo8DWqI/95q2dSBBDwemgMVHrT5x3wVIRdjzg28oM9mj/GNTxOLeQ3wTIPIFNv5z7N6Zk8Dn0n8J5zfAGrOZwlsy7I2x0izca+oSyIjQAotpDIIOpD4kX75sESAxHnvE3McwnkKqLsQdYjTAB8u38AAwbcFNjl6KrRAiWyXC1jILBEeA/JMXwjD6BVsEwRvPxEw1B9E3nJGG3eeT4H/6ZIppkOJKeYHDpcFu5hVV3fV1aWcqggOW/KCrLkMjc8yTbdOrwhPAbDaafNz/OXhQ3uiAOJIN9yGQr3CQjM0vjID3saPhy4yxJketE8NoM2vaMwiW1o82ESh8nL2aVtpaUiN2MFJNN1elOQoFFmZyPorUqs4Km5uAY7LHPHq4z514jbCGnUViO6oZFCizDGbdcP0FfvXOikkzduEYajVph6qvGM1Vo9oy8bHx8PjZ5NHyvv08WTiyZzNgYb3/0QBBtoXrCK4QTMBCDpD/ZJXw8cecIFIwcc8HL4KSX2qUQGx8HQsaQa0Z+HDh8CiLIG+4VxIsWjo11DOmUMKotmyBLiwU6Pv2NCD2AIqS8KMbIOhRXBtktwDTG6DsTAaBnUIIgXZ4TK7YenBMlYKycJ607D4imJCuWlaz1WKWwMsaXYNT5k+l2oydAa+Ltc+tB6SOH7gufRsLBOaxPR35NQSk40sJFar4NxUcyCBfSf1LnLYhgeXRHWX2HyB1mlnoUhrV2+dOdD+Mbg454ZH2ndcHqlfAe5Hbm3RxptGW7iLX8MJiL9cDq94o8Ir3Cgei1bk4g5fEiTniKKZm9y9P7e3bnc3vsza+xMbKdwNMssxCfgUOdDbs2bugWKZTqngES0gVFePXOwC0+pbIw2xMkPbAcC9HbI6wVeoluByRoE3Hh5/P5j0NeKuRqups/TaY9D25Lvbrlutt127rq1WmS1wGnLOh2AO3Oq54L4KECaTKWGc6c1YXQMhNTRj0uFlo3SNuNzXDPVB+3RZXm/yKD4etQ7eNk+GZToNtNlCKIBYUAbGiuUJ4JRXb395/vrVCyInqywtSA/UHuj48LvBcBIcoWa+98uoylasoozdAUjHhztmhei5SWNT9yF2r60C7X6CH+MnVstwHzIgKOg4MOICYJcmWZd1s8g/+4go4/ZaGPFvzxqidhVI2YVAe4e3E/SMcKFCi4q7/ViRc8fCsGjat5TEmbTGtHe0MiCBSdGc8RPyJIDqYNwJLnunEL7SjBC8Ob9IzC4lF6iklHKwf0dPng6Qx8MtgcdtHEJiBI/b0h9eEz6CLToVhC065ZZrEzAZnD3fblHYozWADSrLVMBIJTQn2C2Y4TmXtuyOWYbcKdr5TE0bFt3IA5Zjvb7Il5nDChkhN7pOcy63LXEzd9DDdtzJd3h/8v6WZWsyg6YwzKuM2YaRl3qmG707Rm9Dlo8S6lSiiZ4BxoqoPWF54z/E6R4HLnfTT2HbTICQaWXGqO5iDO5D/Q9c7n27pLBi4S7uXl7c7qH5ox0E36HM6kOTL5fkJoUOBv/0dhDq7uiFWD/qHzXCnS4AzchQ8RWp55uLXLRmmkcnUDEsLkpRap1uakRSMKHaMbZsQSqaUdtSLfmeIDDAaYwfB1/HI3eYSNRKEBcJP0Z9wjJQSRytjF4kGxk7/wm6fB1POnBHs5NW6LLiPdPkixtfn5AeqQDdGvT5cdvTL/1OFW43L2q0+jJ58GH1HFWk3nraCNYzAqHJKrTtMqkJyanAfkaVtRAh1sieoc6U2gZeu94sm9pLF+jGlxfS0MgzbWpDI6tlMzzOcvwJF7OTjquyhOEGRw4GrNWRGT2789TxeG3tGKLVPF+WU189FImRFo7cYh7EHV8RNsvqZcBVpRW10iR1IYKcelu6GmSZ9U3WRN4J2hdgKqSqxjPHsOLBbMvriBwWHG0D+yGhjpOaD0HQidkbzVsP1wSg6FsFFFI0idpjxPV68fEQ6pEHojLEo9fE+xxdDjzgs1ZpdYPCQUPn+5INQZcsygEiCI0swKxDqNiyqZ9gC+q8bhB/hsYXQJDR3fJfU20K0vGorqZw6z+FKGENCJKN7miGa7O2oTrPClH6Oy+ZuqL5SSsDDQjK3iJYm+3u9fjBaRFRA4j5DU09VAVVhnygtqCa1a68uCAA6K+JZnlddxXrpXM/p5Ux8QTidDJx+1mlzewiq+0EEaPpj4bb3j0sbWlyZSwuvFeFE4zkTLw3B8JDI+xQie+XZ+DCOt3xaTbOI0ea9jBB0dNzQFnvHmId3Djp89fi9H8+P/novT979+b9RyCUjxrNKgEzi0B5Vv50evYR/XPR2A2nlGCS7NGaUeMP7FiEDq3ILxEHTcyV7erlp9evvbPnf/fOTj+8f/f2w6nTn2X3nOWzY5mm03yZo8ENSL5eTW4emVZ3gVq+auaTCzPt4i5wjMxY0VX39avTF9775x8+PP/59AOO1RmabkPI+ma9XuawBmZHgv17LkSt0wof+wBw69np8xf/Sw3tLv3y6sXp25NT7+XzV68/nZ16L04/wq9do7qrc+0Ux+DJ1E1QruaE2uRtmQIOJZJex993XQ4ZGKW+Q1baroWu6vHAJQp2wyxxcPE/cxDQI5oWAb0qO64ONRDuYh/SHm2f0bu1rLBiaASN0d8YKH+LNGoG56E2NN2LBqGIDC0aV54eP/Y5erpo5gwLjw+PJ2ypEp9TOHfLeAlr51NZfxECIVyi34vdrfYJPDs9OX370Tv9Bf4l2FHiioxNWfB/idZ06HJItg3q2AEaWq8DY1fOaoS55nQ1zeawLJqXAe7UewPi5us3HruDADczq0qooV2ljKfn7GJTEHuNPjNXsAzopwKyxOwi2mEJ1h4YLYOtmFHYLOX4XugCzzdN+RFJZ/57VoX49AZpsTEaYf2Y/hV2WZeNWzUj7AAlVvLpylDS5sGM2SNlYp2V3C/inzIJgWjD7ogTFTqnW03En7wPSJs9dlqvcKlvKJjiBB0f0EKOq8T2cvJUQJdg4jiQutJ+oHW+ivSciNmIzVT/fw09QprroyME0KH1hu0aZFVChU2xWSUsetT+Y62zZ7d8Y3aoQ2N7NpKMbBv5xiDAF2uyNSFnHI8nEoRAUis5L6LPiD8IUfNELQbhk8dMA6ZI2GN6OabyQ/r30ZPHE8HD6DpoVFZm33XwwOcwnYNkCaxKjr6FrNUgA0Tw4+PvnlovReqo12uTDgduHnkmoh4jm9cIqr9nhQdtHFKr3jJf5Y1iEyf1LQomMySfugDGgFoTXzfmAnHXRNCgg5qIxxYw6tKkFfN4c4xckawoRoK1FBc5HOMitvDjP3wogwkiwixcgghFNlql9WUs38cqbZCXRXyI79Uk2hQ1oCb0Wz8UxcJV7HMDD7EIIMbNyj8OjvBBfoM0l67W/iovYqnDu261rWYKRREtYPbsqRJZh9KrcA2ofJ6voIWIIAjAE8aOLnSK3CGePFaGDBM4xPHA0wAFoPFjbCEJ3/v04ejoyeNHx8F/Ox5A8WFX0FSZ3ueE9kmF/AK2cYXxHtK0A6MBKwqJOtVOjzZEAcAeDSfI5heoDuWlEIlUQjISLYoRIxyaUfS4sjtuZIZXGSpyn0LfG+/v+WUOYkKeehIqBoh6kVd1A//C53JhA8pAoDhsMjp5l8AXfjp7DWURfUnNI0HlqDyp1+ksq8nYWVbAcqDAYldLDc1PEImQGiRsqteizvDxYBAqbi+ZAZlAH1p0gHsyIBmKFmF2kVbpDAThZLqZQ2U1/O+o14SOWIziGCI1bLvb7/SMD1VdbiqUxqTel6Jbta5m6qhgma9gmW+B8nhCPliJOLBF5Nq3yAAgkmu9vIleXv6arS8yoFUwiCo9x1eT29vWDkMX6GZPSgTUr2rIquxub6NQIoW7Otb41ez4c/7wEstnldRuDUxXkSYS2qo6JFPQL7Q5L2AxpyngIPRMTGTw2/EUqHkyrOOIoDe+C5ZHEv8CVRzJh0M9kKOiPSZWJG9uYhvv8QVloSGsHQtFWiLnN2K0QlMdArFZHPqszVbtL+F4ArChR4m+pK3IkuA2kOio9kA7IpoZT0dI25JgSJK6Qw9BOAzLjVV7GdTkQdx+00OcTtCbcG5kEV7JIz1Bb5XXJBkTDdnGeFqWIp9EIA0Niq11TV6zRFJxXLukpPDLLaDAjt4jlqlotaKDC+nLgfbu7HWUY5gkJzM6vvSTtBj0az6l0zEySElc0MhKEN8PVjprPGw/h9ZL6/Ts7N2ZujVe+yZ6DHFI9FEjwxd5RYcH1zVb5J9jxiIclHDIEWY4XjYsmIHTAcJHOBp8ZuMdZ9nnzZnnVWyqHykD0UHbrQsNCnLAUSj6jB4kHOwCGC4DYkpOzP7eAzIefjcJj4Wj2IYaGi+SSamkQvkRt3tm4qhbJiL6XSgRR7s7B5DaVGRWo2GPFTyJ6dgzU7vDZcNZEFMnlmWO6K/fbX+sV5TsomRHYAjUfMuXy6GUGF9OXEUhgA2uAbUDqAXtGaQ71Kx354mLmUecAcxmtaaXTZVnV+JPFSodB6mJ2ORWZsdCVw2opsGhtTG6fjRExjKgga6/tae6CV4kGDDNSm+onuQkNBvQMlf2Gac6W4XWTB5fUpfJIq0Iv/AbMywKxeB6/KY34uPzTG8tYwazQ7QFw87Oai3okCMwPs9sDMYjpSNL8TVA/WBgl3j/EiUcuKf6YNWxdPIAUQQfa2Ustb9A0Xl5M9Qw6VhLxFjNvRjXhT5IP7Tuw1ojw+vZGy3VTiFgdFfof0+be4jBxMgrY4zboaEnCxgnOodHjGj2MM3Go1KQuLu/o/UF8BW7jiuzhnVsAFZEt/HnsdM+EkyJSCR5jduePBq7+H1i+Sj9IxjJkjpD3busNNbeyQoPy+oNO+GthvWx2NkvnIY23gbK1sOI+Q7Cn8s7vb6hrNp4iAqe0Tq9QQ4jnOUSPRi7TJzfooHzacRT8HUbh8fDSfhkIAdRA5Nuqh+eNEPgaY4BoLohXQZQQXgtXXvCz3PTbM8X5xBtDWV9aShTiPVUtjiO2IyoB8YsahLOQf6GfJKEieA/rTOopzyftqyJsEhyKjXbsSMADdBRa1bURXw3tjkeALrpG92umWBgsh1zq09gPShk0TH/7MMYuoc6AWksESBIAPuSV4wBOm0YQqyA62I+GB43dNzQY3IoCGwcEUP/vq7UpDemyPBWXWEUfqbFZi1R0xTS34kdZ0+wgz2srrgHddlWGa1mWy3wtfhhxx1lFz/ctxUtdNBTgBDCXgcZK/rTN1YgQDM12kEq2Ao8aqEcubBrLhSrOfnCdD71nO137M0HO0uiRd6A6G8VXDrOL5X0GNQibsWfvOcU/UhZKby08Mo1a3ZCm7XiTVpdYjgf7BsS7SrUJvDPOK2GslpgSyS91puKFg7VFbCRNcEhLl6OVnXd5hGpuQ7F/qnxDuaZ8abl/IY0hzqtBxwg3Kl5XGURkkMSjfxK/ed//qf/lyGKacFfxt6vzeThr9Vffi386OFfAvoFBVSIzYVQ88W7j89fv/4n/Hr189t3Z6cnzz+cMthz60Pql39HGFS19lETtnblQCxijwqwgkVNYaP+OkQzIimt0PljHWjHVB9DB2Gn1W2of2m79C3ruJxo9TVp49fG8qkb4bD4QB9ot9+xNDoJ0Shhitj3PZDyiv3dSJm0xMB9j8p6NQDOKlWSSQHdbGlQtinrWOAMAcu1Z49vggcxz1ch/a3wu2ttz8nQ/dumhvOesyJV3e4ZKrbkjE/rj3uGAmw+HKTKLha/07DkigAyvPYoJlSTapmBdzbirnES/YRGdHUaMOATbrS1BpP+JZ3Xdjv75mlmmOuRCQ4Z9yA4Zr5yyhw0Z8Uq690b7AjzMeAj/twzMcNZSnYHHHVnkR7EW3mVCJ5phPM+duQDjJXMe0cgg28o9Q2uX0GK+tRwMtif9FLH49zdFzMzC6UTtHZgIqx5bDI+4Widh8I0J/7aJkkUAdZy6X+OY+elYV51NYnd3Zqvo+nlIzyUobR9UIZrphL8hJotGvrQOc3Wt8T1V9HNxbGDJHBuLg5pu49KI1RAWBM1VNJOQsYpyY9AVJMRlDPYZJXXNemLdScCcWu3ee1KkHBqJMyOJTQjIUWbRbOycvT2Vhs0T9JNnS4l5QOaM/vNj6HOFfOv2h9flhV3+fpN+BNQyufF/CcMCz8pi0WO0aduxPWo31SJAyKTHT+y80bc8o2h5Uz4S9SAHNRyi7FG0PvbPPuNhm2DYV8ahn0W0I4Bbo0DpDa6lrmy5g9QhxIExNtL51O/eZF8O80bNpVNiyk9JVQnQQ47VsXiW2W/bABw5uVmCswzN9yuiDwDYIdkTnXZCEX2pOOnZvwdq6uzwf8l6xi6GRNQfIO1iOldKOHrq3QNFFENB7ccD987GzHqMvyLsx6DP9ohNJ+4Sj9TDINYRb87fkyv5HP87eDZ0z7T7sIhEsIvW6+Xvqw3gvEdzldY+JFEkb89feW4FCp2e6Bj4BwJ9AX+tXj19v2nj8QoDjv+M2ZQIkCiJmbbHYTCLISlH39RVblENIZ5VpRYoxA9co+3E9JeF4Dktoy5qHVa3qDdCkARBJYlymG65VAXk01FE7TjoSdz0xGxO6zFuuM+AzY3ey/jMRv2MS0E9kWyIEWiGLOv1oBjbqv6Il1nY/IY2ZaheNgJYp4bzBjihAlQs6EJUG3HELTBTENfQuJE7ABcoIEL2nrUrvSjW459w0jMRg9FGFPXKN/WPSCUeTWmRkGpVC+Z6lXxmDn+Rio91VXmSTRdmzabIB5dXwMQ6nR5ExP9wexq2z90yCvY8RLFSG8moRgXgDGHvM9ap0l/bvv2SkNbcZ5Y77ztLQtGruPoXq36vRwQePiO/0GkBTPriNBFOx3wmJcgo6NsJzBtyEiSb51B9xuO0UiBMQ9kPAhpCsPJqEqvu7XnGakZTaWwvszXva4ieEjLumdaFklbqgZjGWFp+D8xt5bdhjfCjhPvMoZncmIF+SmeluWS3FjMcAJievGomlfkzEWWnLLuPaKGfcLYopw0He2z2O5hS1MQ44vAuur0gXiPq/GwT9m0DVl/6HgYsAWScn2/09KZ3Lbb9BDfUMh3dTMUvxStQnOUKdihq0shvago/3xL2sKPQHj519+yG/oRaJUgN260/rHW++3dNMM/E5urmxD2/IGNFRTW19STuLNW0FuntoPHZN4d9ZJJYkZfu5ErEpzJzBHZYDscJ70zCXOMf2E5zaom3cuPk94OcQDs3nlzgVzIv4kz/4COlXDuTsS3jvH+PZlyOyD7Uxj0dvbUP8p903wxvcMy4eRnqC8zzmB7eett3rR/iv+OPh22wTCTJpSHcaAgPnQRZJf9B/GTffke0Y8wO+Rci9oz3SaeVndxqy1ONHz4cFNgENV/IUv6J++9cDhOolNtBSmrOekzS1GAGj3HLMe5T2+gTV42RD/IXptUwmhftcatdUsLY3hZ/MnMpZTLRFth1TM4RuBuNP11OEnX+kNr2eUcgxYbuIcn3deSwokLTVDuke6cqHuwqds4kjFJYriODjdzN0cbAsXQITcuNaQVC0LzzXn5Y2fgdAQM2/MvsksYGLHLUbM8B4E4qstFg9FRh8fBeDBh2c4PohnmLYFWyXncwdyGVrNycYxV2bH2ScuvPaeux/kkIHcYM519tHxr1mEnsIPa7FLpvRvCWh+1vStMx2VbnDCp4e6NCbuBLMN+Vq9vQLi07SXFpDD/IvkzyfYItRn3ona28i8cjWcdE91UhBKu4GQux2ztmKUskEi+vOZazCeuo3qzWOSfOYkFe6WFipRFYTsJJDxOc8pj4eRQnufpeVHWTT7bnUdZcib3JRw2te+fUFlau19eZTJ/qzvTK78w47Apll0XuD+WXZmJ2r8pv/Kzwd3JlZ0y98n2SzX2EVxob07+vpRL3S4Np6wIdC5en2eBlpY9SZ1xaDrfsky7PxPzk8G98jCbUT4ZuIMzkXk1Xk/hiWbbpGSmxNJkE2oFaHY9yW7vTsbswIoTe2sTMm9snmgOJO5k+uVDDtgpMRFNPmcA1l6+BPGXeTE3IS+Ov+8qLzZNVsfffrdLDWf54b4A0Z5siL1ZE3XjvOnMO4kuBUptMFGBlh/i40FYz0pUq+rM2JgzU0mI8/MG9VwNpmyqbni1cH2F8WF6R9kiMeyIcBLKEitKSanjwEjnU3Nk0Fb2Ue47Vk4O8GcDZSJVr4G/3IWt9m6vbCxnrRdBwripsK6Bk9HrMTzoLIA+lW6q9J4uMfLqEHOYsv+7twLO2uPEHb25znUs1YaS41/pPNVuCnTnYOis1CDG8nHYsYs/HO+2vh1Kph4UaokdByhaznmgU+CGyzrHpnQHDKI/xH1I5oPsqBTa2UYL/FrZSRBCugbJdnFg8dh02X79w3FvXndaRd093lgjicEJ5ugiAuyWTjFdAhPVGRDYbx8HI+bQgCJtUoypbr3l1Hr2E+bmw893+cVLjCX5kbcC9xErwNJoX3eNGigLFeAKgD3hdUXM72MjbIAXpUJBzl+7fANmxGbUkBCPchydrYtzhtzUt4/Dr4ytOPC0mnRHzW1zVagknmJXlbqaHe0Ivwgcc7/13L8r4gBH2UasatjRNCpHpwzrgdYOI2dYBMzVNJ9JjCRbMQHzZvMbV7mMAjf5UNLtLOyqwsEyNlE1csVO1moziO4x5m57PlB7Jnd9QghLDX31bCDwrmXTw1a6eo1HWL4lRzInBb7ajX9F2dR3Q0MoaA2wmojNSDabckOOQw1yehLeiFjmqR6gvT9KIasNYznPKopejXdckzKSpIsUGUAxFWMTsa8o6QQ6CZngpnU7GL/WPktEj6DA//3f/0e1UxESt1Bea35hqJNTuEkmeqM+esM+7sw9welCJf3SHIPGTeO0Yq3EbfJRmEDnu14eG80AZZ3V7MPalKvDWZ95mTFKJh5Z20/MMeO8LhQFIl7WjU3ryHuiP8AgrUeTfJOgcGKEVLoky5HxBjPpQA50xHhs81xzpJAJS6cW6B9Dh508XiZUJbBJxDYrv+nLs8YRtyRhcY3e5BvbrNTWm3BP1mXkJsSZORa6LjLivugXQE3xYOQeefELHYx0ZjHKrnMZD8QVVHsv6oRCI0nKowPbxQCmffmUCWaRhdWJfhBQ0VRkXZWJJVYhS7zmtbZ2EwzoCXaDyH2LhHf3ZOOddScmMb+klIB+wvY7onUB5k3T2e5w4wXU6FiZte4onrcIpsNgB5ryxl3K23YmMW2EEovZnRpNo9bmUUpvf484CRsfEUhUtOAhfYQ0tthd98dYmLCHTwfDKQKI1LmD+YJPP8btMkMNQOaqDrokUIOTw1AZmBqZHv9IGFav5bSDzYYOLuvLeH27P6ygsz3t5K0255MdKb1qIduQ2UlYEItiHsFqPTpuu+9rj4mvCvhsjdNxk295+OmQQCc2UlvNR93SJqXXlk582wdfn5hIq933+Id02KcdEwh6MdiBjeZx+Op3m+bd4g1lZeaMnzTAzsZh5mZYw6Rc6ATOjjFL0R2VDJeMF2p9cogzc0jdNebdSKdVTupECWjx+hzQ8dpM6BEDprnHyHuJyRroOk16MQK2PC3wUhxWHhzpC888PDg5XuxEl2aifgB6ci976bUkuv0bY6K1LwatwCe932JavGeoQc+2cM7SHSdFRzjx34RzIOuS7QzEXdf1drBTby7W/ZE3UgXKIK0ApIXpnECInS03c1xazq/o5JnBa58uMAUbqUIdo841XjgCG7G8AVqYLxvYOEJbNsKJO9HIEUDnsCkPM2wmX2eYtNpD155idhMpZ2jdbJRs4b2DXeREDjZk4Y7iEs2AfAFmoDVrTxnsdKSDzUcqvEH78dExEY9dWWeZ0A2EXmxDyI/Gote59oUt65Lj7olbEy0kVco3lwKaOuqa2PDWs8h7JdnDUDhYlnBWOYsc96qVJhLfpzTH3J6a8QnqSkj3GDGfjnZ7mMLRnYeskfi0OGNmwbY+0vZ3I18g3gRKTLin7xZOgNyKbI7sJhVpAaJNXOLY8LjDNpdnOGfD56H8xXMxdXAaW+n5lAleAuQyLdNq/grPXLVZkzdVu5tcf3I5SvSQ85wv0B4tVFnAXDcVXpwCsh/vYi86Vtsck9tRy0GBUkqOaBt3R11tjVswlh7yXXhxezyozP06tNcZcitOi3mEbldkKzKcdm1gXBPkVjCaodJOoKjhvzu2JxvZprvkSu5ahSZrp8jQWpsn0VEAhaFC8sVcYeR9oGpDWBa3FYU3gKLCc1MQOUR1qqaDTcnidha1RW2+L3gGx8VvKcrrUCgn59akwvbepVYeIY4OuqSLttB7Gii68QZJZwCM6ewmoS7QIWbGgdbwAwnQgt4nNQcMIKez2NSc8gba+XyXVo81Yfc3IWEw7VcakEbaBYvTN7qrQ1SOYi2ot30qaXZSmDGVNLwJYl3WRmudkdbusFoftSY6fAc3B+UR2iS+PnqPZlPUM18wdVq2GrpqkNbsvj5XKV9mwpaZXQYrXhd9Cw7IjoxdD3W9/deCOjyjE6SytQG23QexaRhPpqv0ePX25N2b969PP55iPjarp1BHrDfjC9BolzToIn2Zb2ZEH0ZCUpArIkNLPK660+QkSM4yaN3GZHQTjxECx+5nx+5ILU5GlIx9zIk7WiVtaEinig02ouehM0pz4kI+ceHiOEziO86hf0Oua6FYXNlqGP6eVWUyz8WvfkAIm05p3D22vfUfjQ2bgzktnoTDSdtrghzkcJsmeDeUzvthiKymzDTFIJarb1kbihputA7BEiS0e3gPIHsocHG6MpC2Uw3bOEnGSprfGUxwcQyCLrl2FOsIE7/7i+OA1VLK6IKxRkSMsm9vDAhdLhTOx2aFSu7tyZNfe2u9ULmMf7mXNWrdUR+jhl9oBYdflNkwPTjzAh1H0N8Ct1F/5Cf+YuezOMY3bIrNeW/aWUgkHPGWh8GYnLn9oXrNV/K2/NrkwvOXOuHV4VbCKxu07s1LAtAbtN1SOKpGcRjAB1zcy9NfTs+YEkWEVJxj62ZLvxy6kNJJ/mGS/RbK2XFna83tYJi9Iziwfp1Dcig7yxD8mS+UUCpWWY+Apub/2GQgt2C/GGNpGEe+aKPc1J6A4ZEJsK24vYjgG38SmvZlbWudYrZ9O5sROEaObMLVQ8lYppNL8njxgkX64VjE2W2UqLs1NiRcrEPpxZ78AZeeneXqjloapk+8PwhrG+SvBfeyHOPxfQPsa3e+ySiqrGVSntVXdxFy8gTZYU3GSe2+V9s6KPS7fUj67/huHkc7d8r9CWjSMI4yFDErTbmeMSsnY7trqHEmkbQwGFQoUOR7NhDUtaOSjahW0g0ZlqAX4+u4y55EIdxsUcdiLcaCp6Daqk1tYnFSbW0xEvtsLv0JaB0iJKrIvblkjQHx69J9WaWf04vbZpvdyZ18f+6oeVO+3HKDFeaSirvSOkLjuD0QS5qpIiDT87K6iX3XUmrTDIjYrYwqQJy/OwpDm7uYRtU2jJjLTTgehYVidbdyodOKzFFCQg22TjBUN5GLgnixqJzArBlpWSUsunBgCxZP5EzNlcTm4EubE8i5x4o5snsllDNZxId7F198GB0/4WHnshk9YTRY0w/O/YQKd9k0+CK/bKqibv4HvHNC+pIL4mm1tE+mTQbZukirb+C2uuM3LE3biG+Qa6t1w5o55moEePfnvtCrISFUOpIXhYkhjoT8GB1rvv3ing3dDq8T9EJEAFgCZdYOeQTk1dStJql08dSBg1F8XXYXyQMioYyjbvsiUdQX0hV5SpI1G71dFUO16AVs9t/p0V+E5BxG2J3TOXO348EkEHoqd+KhepLMfO5LliGpRmBYEKCy+MKrL7KssZnkDetQDx06zXN1EPn9SL0Q9rpLwcWtbVOwJT6hnLHL9Carap9Oflg42USOf4iLH0hdxa7tYj+TBE5VRBX3ZQOgAhzxp+VNEgq5OURgMB3sOxii/Mjia3JepXOmnltVnDzKe2ujfoErT/HaFNtA7xTGh8VwMhQiRr1Rrft04Thnw0pSNYBcvkVPRoq3FwFSIh8+RSEb6DOX2MbhjGxWIEAj47z0g33LxIJzayCB3GL7dQ0GtwwRQNthJkWWyAJZzsYBkfhYOxCu82XZKOPl+GQQOn5MdZOtoexgMAjMfdZEuufUzSH0g+oK46nH5q8ZeqxoFw/bmod2SDaP8l3WOZsJ5t6zp0fPnnrkXEVcJrN7Hq0wJacvF801cN4iUdDIWzqSA53CxnMYbMnRZ5sB2R85ZeGDtJhxCHyWKSO3Yrsej+G2SuvfEd0UfqSQn4r/QtfPq/MN8nq1/nIiGd1CzDJ2Ui6XKYzn74AE33NearRck6dbz6giDiiqkk2TL80YMdcvAYHdlP1KusUxi8IH3Uisxnrf7WXf+zLndOHLOA92P/Q6SJ5YeCKtNKAn49LoeBXit/Os0r7TBDHm5kiBeUrI3JuE5uSixDRGVA5HRgWDAwYTolwkZ/gsaNDbhAqLPtF10uLONE9vy2Ob9tpsmMZWu5aN/wONO5XaPdhsXzwFy9lK3hwZguXTeV1eAu1/WzYv0THC8Z+1x4rOLcrFsr6ICNh6DzgbKi3h3H7IMg851jenpJEzx/XwUJDAYeeURt4LVgvUmykgkgaQk+d6tkVmOxwdr5lbMILJOB9kagCc0sT9FcPAYhGSiu/y9NdN3x6YGwY03mJAEtdboIalltCyIxSoibPhhcsr9BeHk4fm6IOtE0G5WbfOD8YPHcO/sHezLOfgZFqK4Oj4KV3f0lQkAN01Bap0O7q6V+ErmiqJFVj+z1c6CZLPL/5JL4I/6wXsOWYa9TlAe2RdHSWrEuDqZbp2DzLAPR1Io13nyf4QP3vadzUSHWIC0hkx1YgpOEqfzrUBZGWsLNRLtMHEZ34glB49fFgxzL2Fyg5aPsALeP1XOD1lRUTQ0qAa7w5DVQQrQ9Bi00rzefb8Z3ughMdme6q+xp3B1O79kXaANax9y5PZTAr5aykJjCZO7JEiIiDWSMsTBNoXOlpdYvgSMBd0Ew2xO4QUkvJS+2HeN4hWht8f0xoceF8dKXtXw4Y9i/ewyO7sRXNm6rVdtcV7l/4IRCSiR96GB/2FAYILb7lTW2TVqrpVTuMuFLb+NedsWTKH+oNI2+O0LuUC1wE9W5ezC7wigIZBzAJJ9inqgB9nh9+xf7T2xMZYGEW3MyQ1wAW0ESpkcXNUZqH6dbVZsj8zfuGra5J5NktByo4Gx9SJ4zuNKSlbaA/7PYeyijAgxhd1kGJrZowmlQACOYWbbQ73+Hyjz3O/nzMhI+0WYiwRWs/YvWrpuTjUCuyidVZqsknCXLDUfn3gSa4sxL3IBKIhi52yiK2LyONSPpDScTton70TKAiSY+owIEoXhN2ScibU0dzPQhVNSeAcUbepW83Ze7qKTdFxPhlhidgJQbZ6mspxeuMI6nuFI2OGAdIiUto5ubVuIlajiJKXYy+iwjGriMV7XZX18mpv5fDeqFFuGh50fIk58QkIQQ1MYo0OSr/ThUIZ0SvjjLAVAYYWBeHydWA623oNDZim1Ug3wFSp9vCWZ4rRQq+GoyXe8sTXT0UaUtB35nnzE/kS+x0xoh9AAnPl654QaayFKdAJkU2z85xhJEyrc7r3s8nIE7Eql+HDh5fX+Dro86XQs3OcO+0Vz3x967KckttCttaXO9N7BAo+xTwIgmUH+ToI1Hwi7Nsnpcfm5bhfip84k8Ze/9Cc84XX74XyILZUebhjZb5iRbpj1Xf33D1SUqjpgdCdtK2Ba8/TP75HRCzkLf3WtoCtBviKH7kRqP0p7ja7a83o69eA0s5h7rvDmfy4HTfuPr9MMzL26k+0B/a+MQbGPxk2ClHGZjmnHMiEMEadD87cy7UosEx+ASlr4WJZnu8ECbpyTxJ13OfoioZg/3pLoZgCTGFzv9w6UOrYwrHwv+FY6dPfc7CcT3etvgxP2I5s57j05O5eNAcrGZvdvdZNq2G6S4fw8C+iXpctYh+yfQOypV23WYfbqskGZx4P5crurfUlMRRHF2/pu3xJDoUXp+zqJAjpfjvC/cyKxnjBl8OFxsSEupxkLIwkek1ISkahYIYvjb8Nna8k/jsfvw97WVYRxb8NF+vjpywdoafTFU2VGdNY86XyFstv8aNyI3qnhrztrXHAIV4Ja4k5Ld9jPMHnGg3WFBoOXHsMTDuyh4l+YAM/VI3HZA2jNJfTDC/MJDth2iDl4OmIgweI+k6J2PGVOKcQjyrJayjQoLVFL4NZMTt4c4Ve/7eEoRZkrA2lQ4MCVYqsODFXtzQLHCx0hzCAue2Rhg8kC5GkCvXXNluH668jCkx/HbEhExMtryPHiB1B55K7RXsljQfhcfh4EsJSV+hUyu2pLTenW5Eyafr015cIKRIuaVZ0Thns5syox212nvkXgpPeAsTFAPM4Y/cJ9v+xXDauOe3zTBTE8Q5tsW+rcDK+BOC5yeHkJ+Ui/p6j9DDnqsZB8hzOhH+EdWmxlwEHM7ODS9yjWvZ3HuagLT25nzr2do7EIHHIhOX1xtFZW8EdAXWyY6wg99nzlZMiO7hOT0s8CnWdLbT2Y9z+ZHkVRME87LZMN+rgZTjPuKvsrSuxfKQIpsXDtwHFDXEnePg5CRF/GtnEOfTN0Y9wgf44ho7AGnbVGyJB75w2iPaIFlzy5qgU2tUcFGMLhbgaeX3RjhLjGNa9LvScyWxTuWYdGq1141faBUoTGvNJe0W3I4olRrbtXt2GHh30YCeg/amJhf4DTtV7nfv/gFs/efNzD2zAwdCXunGCGaw20W7NPRz1v9JFvwvWht3QXvr38cBvu8G3HeBHmPhFIIvjUF1Peq0+wbYO/h+I4kdr'))
expected_runtime_sha256 = '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06'
assert hashlib.sha256(patch_source).hexdigest() == expected_runtime_sha256
manifest_path = PROJECT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest['files']['src/fever_colab_runtime.py'] not in ['f2ca840db6e6d871bb42a1bd3ab84ef77613d7946541add86c407b7c4b26ad55', '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06']:
    raise RuntimeError('Unexpected bundle revision; use the original supplied ZIP.')
(PROJECT / 'src/fever_colab_runtime.py').write_bytes(patch_source)
manifest['files']['src/fever_colab_runtime.py'] = expected_runtime_sha256
manifest['notebook_patch'] = {'revision': 'colab-debug-v5', 'runtime_sha256': expected_runtime_sha256}
manifest_path.write_text(json.dumps(manifest, indent=2))
importlib.invalidate_caches()
fever = importlib.reload(fever)
assert fever.RUNTIME_REVISION == 'colab-debug-v5'
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print('Debug runtime ready. The original ZIP and earlier experiment results were preserved.')

Mounted at /content/drive
Found your data: /content/drive/MyDrive/FEVER_Project/FEVER_Colab_Bundle.zip
Verified bundle: 19,998 evaluation claims; labels separate; pilot training/validation disjoint.
Project ready: /content/fever_colab_project
Verified bundle: 19,998 evaluation claims; labels separate; pilot training/validation disjoint.
Debug runtime ready. The original ZIP and earlier experiment results were preserved.


In [5]:
# Install the checksummed paired-evaluation adapter.
import base64, hashlib, importlib, zlib
paired_source = zlib.decompress(base64.b64decode('eJydWm2P20aS/q5fwfhwaNLmaEYzSbChlgYcZ7IINs4ZjhPgIAgEJbZmeKZILpucsTKY/35PVb+QlKjEuwFiDfulurpenyryxYsX79O8kZlXNfldXqbFpZKF3LYY+VB9f/vhY+rJh7To0javSu8xb++9qpSeuk9pU5E/yIsPb/7hbauylZ/b+Wz2S+W1TZqXeXk39/CwrzJZgLqXpW3qZdVjWVRppkDFy/d11bRz7+O99KrdLt/maeEV6Qbrd3khvVzNGpmCtbI4eJuD10haD8JLb3svt5/qKi9bT/NL3HVKKnf4RV1VhQfW84x5n89evHgx00d6/6eqMtyn7X3Y5ns52zXV3qvxWOQbw5X3Ho92+U4+yCbZVuAtabqS9nipwqUbOZtlcmd4SPrTfMtG0lRVG0QzbzQQE/WjNcttWma0Xap4tV7WTdVWOJIeZt4OAtwU1faT8vLSWy3C6/BmDaqYKDLZxCNSl75oqo1s2jTZdUWRFKlqxSvVNr4mEQTYaOnHJIs568T3NbVLYSfnNCmCOakhIQX7vHdb7etCtnJyr508s1e1uGFCwo7dDqdMJcB7/3ihubYUV0K1slZiHQSXgm8MrTA9fZalPmSrP+6YkXznZADC26qWYv1VLEhiguy1n+SjkpKmF4tvb775djzda92s+fqbm69JNx72Ken9DveRt01TNb74rZSfa+1edNCF1RusD5ZNNMQJb125a6T8A6vEeiVImYnWIx1mjAIM8U1X4q6oNmmRkKRo/kR2Zxj7aBmBJ2fMyKXZSp61zxW8ZXvPzPXxALbZsGk2ZJWGgaK6S+5z1VbNQazpJoI2JPt021TJbiFoaQOT9jZStfE+/ewPCIaf5CEu0v0mS70malZHe9dGOGXVeuS+81xti0pJn2idLA4NRzSZ7GXb5FsMphuVtLD7hbxYXAdnxPE9tgyCh6d3I3whwpjDt/eevabhip16dCbFvqS3Z/A/L9O9hJmdGLm+gVbbn3I1CH1H7Az4neKsNwVmbmC2Uxx+8WXO8Pp2YD2yTTn66zCpPOSBfLeTjSyH12Fmd5SDXGTgJ7Hk802kmwwaX8wsBS8XZudpXcsy859GThXp33Do186momlDIxEM18Mm1WgtD6xDYZOr9s9oqPRQ35ZjlYjIJPg50MT5Vv19BqsGwglwAMc7pOfrb74VEWWoOR4SyqeDWGiIuqxhg82ZjWfSwrPNBWmTbgoZZ/m29e0a5DM3M6+r2h/GsaBPb1YJ/WpQNS6eFoVfx7Fbu7pac7ipKYa40SD6k3CmZEuIQRmLQ9A5IMB5xEv1hywBNw6yUWR6j3mJfMIBqTeRcTzym9WkVazDi+YoMlOKaWTbNaX3JBxEsVEoEudgisdEL35cLL02h3fv0gdceCcfwbtj2hzCStT/iZ5lKK9nX+hbiUj/hsLirKQPuglwUwYLQi61fIrox7RQ8nk22+JWyvvH+99u9xuZwQp8No0PUuUZ3NeOUsQCvvoVLua9w8V+fue11SdZ5n/AbvYyhbpwT4ZuP76/ufbe/vbDGwSN9h7Xau8x/fb9b578LLedxmqzH+RDvgXKgq5U1TX4k8/L24MH5AnRgo0MWTSlaFLKxyFKdWbq/bqHCc12gAItJ1kdNU3s2ZJ4Gfk1OKzqlGaCb8QmQwfVcNycTJjho0dgL0lwwTZJfIhrF9rgxjFbdTUkFMzdEje7pMVzDktx/+e8rYB3uiwlAyTa4ArjmjJBFRXiMneNVCr+BbibDzGgtOz29YHuX9YYxJKuaBkrsovA17GGUm1a3kn/KiwkQClRDMIbk/Y2lDViHlzx+oj/fXVzTVTICdPy4NNG5tgpdG64/BymGSIZ8AwZFE+r+GPTySB4jSjCfHwmHvggfeZEpri1EveIFVjBVsqMVUO2DkIXTBrFxj7XScKIKYvHfPl8TlinTI05CVvg9S2bRcwmHWqnBA4sVdWoWNQgOVKDp6scQ7rZAmSUxl44rfjmIvc5DL8cKNN/+dLwFcw5FugVGqEu96n6FJv5lUjblqyZg4j6JNbzrlT/6hAdpX+xYCa8B7hi1cS+pvKS1gVz1e39RXBJD+bvOTx0X/v7vIzNRm0KNq4O7lGW8x2kQcemxbysGngH5Obrk8I6vg6zfB8TzbqDFbOF4Rd3OdTSF+xHN9eC4TOZB5tLHF951uBgOq/j3tJo3OeJy8ub61eL4L8XV1iu5cdxBFE+3x18a+ShcO6XsDJFqAega9zQt4cMrDkI26ol86OnwdkBOwUH4LKeA9XCCmSJ/30tn0BXbijswEDi8vKjzO/uW+W7UEjaNqnBja1sWF0vNV5h/KHHVsM0vg4Yu4znJ1M5IVu90ca5mAU0WGcnDC7QqOivV9MySm98sYRAEGA7/bB/8h9w0fHZnIAxA0/PFLmD74u5SnfSuI0IxXyTAwIEa5eshweQ4il2jKnyqVSzDBmf38nWp4lgxM+QGjvcadz4kYibhS6/c0BHWhmk1AvX0RhjTWMdYx61VSBoQC15o/ymelQhFdVh1bWhs4DQ2WkIu+yoZl/87SpU99Vj4iI2B6B7CZPdyLSFjVFtpeLrqxAgI0HSSuC7+b4P7Says7eCP9co0P0BMKAf8QfYR5avynwL62PVc3VL3PLqS0FZ55KxZ17WXasYsxVCgytyE1oboFr97rvv/kb6omfUi5YqDdGyJ1hsnol1X+fRwme7c1o1pqc0yMmN/FeHMcW6kZ/TbcsFsLf4LuTzbftHM2vYJBMCgPMbnPZEXISCJYZfpLVPcFeqmrcFQM/zmL8pTDiovQeM7aCaDol+l8siswcbnf4dseqUzq/QLe3cdBls19t3KMo2jBRQNz9IS2KkYi/XhRppmkGNT0+5ykuENGQWf7w8hIly9BwP/30xeS88p+54JC7NHm/RSZMo0cnEmk4F+7TsqEuATOt/fR0s9SilwOFUQgAc0zPPxUcXb/46cM5cT1G3e7Rd9j54aZtUQF8ZpVcx80x4cv0rR6FJ775ZXJPizUzf1JoIy8dg/JWwBBCviBmcQVtHDAbhJKVhOMdu5/jxE/K9iwKiBwkZQ1dgfEYUITfEYE1JRlkUwzaNDuA7IfiafSYBXKxb6mtNVmIJ/yRJ8DzrO3hPpn0VGRpO7mxpU4HQtnZhJik74cgHcQFZVPWey3M2POZVsG+eYezPg04oBqVFH0SJKEMnFC1cPP6R15wLVLg6m88mdTawTJRemjBcBwHsjupkaJ5YkJmIvsZf1uBc3WtuY7rYbphv1Yd6TfcLFYRTyoe8qUoSo1k1GPE1MUQB0tsvKGBQY124ozxqqvdRasnBY0fddoXNxPtgNtTFcokCsa06yCwDklbtXJCRwCKOQAJX4ZKyyCUbDLTrinYGIXeyqRtqrMRP3OA53QUjy1mFSlzSkvDJmq/rJNgzRTRigdzBNBhFRFufx3mftQ+++czHJoffVBQf+dxB7e2MSPciwmHYIdxLDrpaL23PK9lWEE6cl0zCPF3pvjryhU0bETfGJcm4L6VGrDF25XA2LYlLn+IKaOmsichjmiUGMnMnWn7OFUKlKSTaJt2OetZT3WrezCsZMBFxJEV3DkX4wazOknaBfjpec6oprB9qX8M1zeNEjv81fYCh9QI47kZSojfxyLmZvomRsK1QmCfTqCTEYaaDOL6OrNT00Opqveqd1/ENSGkXLKYXnCs935vKvzh4Sl9HM2wCgUWVTnuUD5vBCVTadUgwcSw+3L754X9Fj0LsLTgGD/kH7KQ3F5kOkcfMH8/+u5zb/QZIjS4wdoZX8WI58odXMeEtbmRtZMGX+umX39/8/NMPE9ei9wAopyPja1aXsLe+fWfmIjYnX3xfoax23q8gTLLxA4aAJUweYjCoc85cBEtbwJHnzbw9kCmVxLoD1ZXv9LOODWby0sQhAxOQfo8HHPYNrTyyeCwZisxGMuOIcYTvR08TQP9kBKLRHYCGIf9S5z6KU1zYghF6tTivZbPTB1IzaamNLMZFqFEnIJQUsSIWaoxDMUyhXzilIhDJ9FN8tQSApFjXNgeyJiOoOZI8aYVCDpXbfeHNXIlQ96rcMJlmXYl1qOFNbNANg5q4bw3MPHfFYdOwz6GDoBpSoCOj6tM+YzMdF8/wyQ3JHGi9EaEzppgLSCPPvqNMZqLV/juhVezh2BoCDgSWERP9ra0aR5/Qw4XR0evYVAcvv72KNiRis+cvED+mXsfjNZFVJU66mMLtTqfunPo+VXLSTI7kxWQSwzNExY8waJcy9NAoQ5gowbEvHiEiSIqcpq+EjZIHdbBzixEjul13HJE1cZXflVx8+eaYYEnOqjNvbyX6C4KRmWhFm/z5n2bjM+n4S/OxS8hnUtLJ4ETp9pbVO0yiRhImbivDKYfaIXdcDUe9Lo0Go6E2QxPIIxfGsQzXILDctVuDScvqUWNRU4Wc4rdTTBBO3Dg6GQpP8mRkbrcSNnE+O1GeTH1lc2pkpUwyQ1qy35owitGDobg1R5Es6T2Sru1TYPHMyFALUQOgrqbXI77uwRvps30b2ueM2eAXGOkJeJm8Q48LuOanjecwAE1NAQD+1sDu0xWY7rSPdo0nzrba37HfwLgyjk2N3EpqGWiNUdPHwQe2oUurQO7Mazn+FxHkt0vehhK69U5GGxsJr4UOuk2Rq3uqY2TO73joDU1v5fORf7MIj9NA3gSDNvFxIfDljh4OFGRS+pegHDp/IglxKTM8fCqu2gI1dmozR7latdebnRm1ArRLWJ0kNiBzO7EfHhtKfN72RhdhsOj/Z+GDTNwIkZ3JE+/f/PQhefs/797/fPvx1jU0xCPAnMU8cN+JXMpp7PkMKoWxDMGpwTHjx1eLE26ujBeOF76Ob0zg/aC/3LJviO9R0JLHKn7p+CBdluKwgaykliBFb7daA7HZlJyR02uEPb/MnhnnH5Wapj1qwZud7eFb7z3K1vLUuilPkbCY8UuxuvX+KQ+bKm2yn0iMTVe3kfnmiY7I7eAQJAIykkW7GbqT5YXNXN/DvgrVF9U+PbTzJYvQ8XHLPxRiU0VjQzZM0LUc8KsjLKGXouSpSfJKRJ7u3NHwMW6h1rshEUp7TKyJBZYN7skVh8hgOIcQtDgjHdf1l0FDRMqg23TLC1CUA9ikV+tLhOY46rFx6WL62QaiRSI068RShEe1g7g07TQDA93ncINXDrqe0W8a+JtG3+Eq98LjBX2Z2bTUj7PJiCDlQ86vo0Og/laLwXybpGyLXffVXSGL2iq9k3N+e330BsFsNGCM/j3iIzQk+TNC/WbXvT4wm4OvUKWba5mhmf6GU42+DuzfYwy7hHrh9JeCLE4FpEOOAuE2yFatiK5C4TqA9GHoaMY1A09mmMhjU8Ffo6vnUePn6PXJNKfjfiYZXhpu4tXwiv9mZhrf97TTtNZhJT0DLzfTjY4/eRdjoptBll7WNfyBDK5vo5gRWJLG6SA/aiWtxteg1ppevCFW/mIxaNOHNGNV6qBpTmSA5Eia9DKt6PE+vXJa8cOVlubAEPjGZGQrMLdGypnZzyieRCkiduPQ9dbZraZ65OFJ78XJmi6lP17QRfr4kx3XCtGdUOV6FZorgsv0O9yTbrcdstAh6T9iSZBat2ADXp6Y7+GixdXVS+uhDNkV0Abght0u1hfj2avxbHCmA+v3HNLXWjkMOdGhnJM8/ZOUqKQG+Mu+bTehlB0m6/a1Mu/hw7wk641RkS+P4sj/A7j8dJk='))
assert hashlib.sha256(paired_source).hexdigest() == 'ff4f5db32362711f4389962da410f32b0a106baf7c94a02909a6c710e4c2587a'
(PROJECT / 'src/paired_roberta_evaluation.py').write_bytes(paired_source)
import paired_roberta_evaluation as paired
paired = importlib.reload(paired)
print('Paired evaluation adapter verified.')

Paired evaluation adapter verified.


## 4. Recheck validation selection and set evaluation output
Selection reads only the training runs' validation histories. The expected file
hashes were fixed when we selected the model, before this full evaluation.

In [6]:
EXPECTED_STATE_HASHES = {'1': '982723bafdf2a3e631321606bc427b4c87abc6e2d1f386f45c27866efb9def2a', '2': 'bfe5f5354747fd25a5e54100cd06f17aed6a08104b4d8a9d7313d2204716fc67', '3': 'e2fe1583b3c925e84871629123096f63f20a3600ac6b92867a9ac45d697590cc'}

TRAINING_ROOT = DRIVE_PROJECT / "runs/roberta_full_unfreeze_v1/training"
SELECTION = paired.select_validation(TRAINING_ROOT)
for candidate in SELECTION["candidates"]:
    if candidate["state_sha256"] != EXPECTED_STATE_HASHES[str(candidate["last_blocks"])]:
        raise ValueError("Training history changed since model selection; inspect before evaluating")
assert SELECTION["winner"]["last_blocks"] == 3
import pandas as pd
display(pd.DataFrame([{"Unfrozen blocks": r["last_blocks"],
    "Validation macro-F1 (%)": 100*r["validation_macro_f1"],
    "Validation loss": r["validation_loss"], "Selected step": r["selected_step"]}
    for r in SELECTION["candidates"]]).round(4))
EVALUATION_NAME = "roberta_paired_official_dev_v1" # @param {type:"string"}
SESSION_MINUTES = 180 # @param {type:"integer"}
RUN_PAIRED_EVALUATION = True # @param {type:"boolean"}
if not 1 <= SESSION_MINUTES <= 300: raise ValueError("Use 1–300 minutes")
EVALUATION_OUT = DRIVE_PROJECT / "runs" / EVALUATION_NAME
EVALUATION_OUT.mkdir(parents=True, exist_ok=True)
print("Selected model: last 3 blocks; validation macro-F1:", round(100*SELECTION["winner"]["validation_macro_f1"],2), "%")
print("Target: all", len(EVAL_ROWS), "claims, TWO models, same retained RAG input.")
print("Results:", EVALUATION_OUT)
EMBEDDING = fever.remote_metadata("sentence-transformers/all-MiniLM-L6-v2",
    requested_revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41")
print("Only this ~0.08 GiB embedding model may download; both RoBERTa checkpoints already exist.")

,Unfrozen blocks,Validation macro-F1 (%),Validation loss,Selected step
0,1,66.2971,0.6809,11000
1,2,68.2425,0.6491,11000
2,3,69.4235,0.6346,11000


Selected model: last 3 blocks; validation macro-F1: 69.42 %
Target: all 19998 claims, TWO models, same retained RAG input.
Results: /content/drive/MyDrive/FEVER_Project/runs/roberta_paired_official_dev_v1
sentence-transformers/all-MiniLM-L6-v2 revision 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 weight download approximately 0.08 GiB before 4-bit runtime quantization
Only this ~0.08 GiB embedding model may download; both RoBERTa checkpoints already exist.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Run / resume both models — enabled
Live logs show the claim, article fetching, embedding, both predictions, invalid
pair count, progress and measured ETA. There is no JSON-generating LLM call here.
Evidence failures remain INVALID for both models; they are not converted to NEI.
Three consecutive evidence failures pause the run for review.

Initial model-hash verification reads the saved weights from Drive and can take
a little time. The 180-minute budget pauses after finishing the current pair.
Completed predictions stay immutable; a new session resumes only pending pairs.

In [ ]:
CONDITIONS = ["roberta_original_rag512", "roberta_last3_rag512"]
if RUN_PAIRED_EVALUATION:
    print("Verifying selected weights and loading the paired models…", flush=True)
    CONDITIONS = paired.run_pairs(EVAL_ROWS, PROJECT, EVALUATION_OUT, SELECTION,
        EMBEDDING, minutes=SESSION_MINUTES)
else:
    print("Evaluation disabled.")

Verifying selected weights and loading the paired models…


paired_roberta:   0%|          | 0/19998 [00:00<?, ?claim/s]

[16:13:09 UTC] paired_roberta | session_started {"log_path": "/content/drive/MyDrive/FEVER_Project/runs/roberta_paired_official_dev_v1/monitor/paired_roberta/session_1793981589170045863.jsonl"}
[16:13:09 UTC] paired_roberta | loading_embedding_model {"model": "sentence-transformers/all-MiniLM-L6-v2", "device": "cuda", "dtype": "float32"}


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Token indices sequence length is longer than the specified maximum sequence length for this model (727 > 512). Running this sequence through the model will result in indexing errors


[16:13:24 UTC] paired_roberta | claim #91198 | article_fetched {"title": "List of NFL quarterback playoff records", "article_seconds": 0.5, "chunks_so_far": 134}
[16:13:24 UTC] paired_roberta | claim #91198 | fetching_article {"article": 3, "total_articles": 5, "url": "https://en.wikipedia.org/wiki/Quarterback"}
[16:13:25 UTC] paired_roberta | claim #91198 | article_fetched {"title": "Quarterback", "article_seconds": 0.81, "chunks_so_far": 202}
[16:13:25 UTC] paired_roberta | claim #91198 | fetching_article {"article": 4, "total_articles": 5, "url": "https://en.wikipedia.org/wiki/Starting_Here%2C_Starting_Now"}
[16:13:25 UTC] paired_roberta | claim #91198 | article_fetched {"title": "Starting Here, Starting Now", "article_seconds": 0.35, "chunks_so_far": 208}
[16:13:25 UTC] paired_roberta | claim #91198 | fetching_article {"article": 5, "total_articles": 5, "url": "https://en.wikipedia.org/wiki/Electronic_Quarterback"}
[16:13:26 UTC] paired_roberta | claim #91198 | article_fetched {"ti

Token indices sequence length is longer than the specified maximum sequence length for this model (1172 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1172 > 512). Running this sequence through the model will result in indexing errors


[16:13:27 UTC] paired_roberta | claim #91198 | embedding_batch {"embedded": 213, "total_texts": 213}
[16:13:27 UTC] paired_roberta | claim #91198 | vector_search_started {"embedding_seconds": 1.43}
[16:13:27 UTC] paired_roberta | claim #91198 | evidence_saved {"evidence_status": "READY", "supplied_passages": 41, "preparation_seconds": 4.44}
[16:13:27 UTC] paired_roberta | claim #91198 | roberta_inference {"input_tokens": 512, "untruncated_tokens": 1172, "truncated": true}
[16:13:27 UTC] paired_roberta | claim #91198 | roberta_inference {"input_tokens": 512, "untruncated_tokens": 1172, "truncated": true}
[16:13:27 UTC] paired_roberta | claim #91198 | pair_predictions {"claim_id": 91198, "original": "SUPPORTS", "selected": "SUPPORTS", "identical_context": true, "identical_retained_input": true}
[16:13:27 UTC] paired_roberta | claim #91198 | claim_saved {"prediction": "PAIR_COMPLETE", "completed": 1, "total": 19998, "invalid_total": 0, "claim_seconds": 4.76, "recent_mean_seconds": 4.76, "

## 6. Progress and final comparison
Final metrics appear only when both systems cover all 19,998 IDs. INVALID stays
in the denominator. Metrics are label accuracy, macro-F1, per-class precision,
recall and F1, confusion matrices, and paired counts of fixes versus new errors.
Live revision citations do not constitute an official strict FEVER evidence score.

In [ ]:
for name in CONDITIONS:
    folder = EVALUATION_OUT / "predictions" / name
    count = sum(p.name != "protocol.json" for p in folder.glob("*.json"))
    print(name, count, "/", len(EVAL_ROWS))
METRICS = paired.report(PROJECT, EVALUATION_OUT, CONDITIONS)
if len(METRICS) == 2:
    display(pd.DataFrame([{"System": name, "Claims": m["n"],
        "Accuracy (%)": 100*m["accuracy"], "Macro-F1 (%)": 100*m["macro_f1"],
        "Invalid": m["invalid"]} for name,m in METRICS.items()]).round(2))
    for name,m in METRICS.items():
        print(name)
        display(pd.DataFrame([{"Class": label, **{k+" (%)": 100*v for k,v in values.items()}}
            for label,values in m["per_class"].items()]).round(2))
        display(pd.DataFrame(m["confusion_matrix"], index=fever.LABELS,
            columns=fever.LABELS+["INVALID"]))
fever.monitor_status(EVALUATION_OUT, "paired_roberta", last_events=5)

## 7. Inspect a saved pair
An empty ID selects the most recently written prediction. Reference labels are
shown only after the model prediction has been saved. These labels never enter
either model's input. RoBERTa has probabilities and retained tokenized input,
rather than a generated textual explanation.

In [ ]:
INSPECT_CLAIM_ID = "" # @param {type:"string"}
for name in CONDITIONS:
    print("\n", name)
    fever.inspect_saved(EVALUATION_OUT, name,
        claim_id=int(INSPECT_CLAIM_ID) if INSPECT_CLAIM_ID.strip() else None,
        show_reference=True, root=PROJECT, show_full=True)